# Module 302 — Label Flipping Study

HTB Academy, AI Data Attacks, Sections 3-6. This notebook creates a small synthetic binary-classification dataset, splits it, inspects the clean training data, fits a clean logistic-regression baseline, and demonstrates the label-flipping operation introduced in Section 5. It keeps the clean test labels unchanged and compares models trained at multiple label-poisoning rates.

The examples represent sentiment classes with two made-up numeric features. They are not actual review text, email, or production data. The notebook uses 1,000 rows so it runs quickly on CPU and does not load external files or contact a service.

## Read the notation

HTB describes an example as `x_i = (x_i1, x_i2)`. Say it aloud: **"x sub i equals the pair x sub i one, x sub i two."** `i` is the example's row number; the two values are its two numeric features. `y_i`, read **"y sub i,"** is the class label for that example. Label flipping changes `y_i` and leaves `x_i` alone. For binary labels only, a teaching shorthand is `y_i' = 1 - y_i` ("y sub i prime equals one minus y sub i"): zero becomes one, and one becomes zero. Section 5 now applies this binary-label rule to selected training rows; the original label array remains unchanged.

## 1. Import the small set of tools

NumPy generates and summarizes arrays. Matplotlib draws the scatter plot. `make_blobs` creates clustered toy data, and `train_test_split` separates examples for later evaluation. The HTB cell imports additional reporting and Seaborn tools for the broader exercise; this section's displayed baseline does not need them, so this study notebook keeps the imports focused.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_blobs
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

# Fixed randomness makes the same toy dataset appear each time.
SEED = 1337
rng = np.random.default_rng(SEED)

# Reuse the lesson's dark background and clear class colors.
HTB_GREEN = '#9fef00'
CLASS_COLORS = ['#0086ff', '#ffaf00']  # class 0, class 1

## 2. Generate the synthetic examples

The mathematical shape is a feature matrix `X` with one row per example and two columns, plus a label vector `y` with one class per row. `X.shape` should therefore be `(number of examples, 2)`, while `y.shape` should be `(number of examples,)`. The centers `(0, 5)` and `(5, 0)` make two visually distinct groups. `cluster_std=1.25` controls how spread out each group is; a larger value creates more overlap.

In [ ]:
N_SAMPLES = 1_000

# Each row in X is one two-feature example x_i. Each entry in y is y_i.
X, y = make_blobs(
    n_samples=N_SAMPLES,
    centers=[(0, 5), (5, 0)],
    n_features=2,
    cluster_std=1.25,
    random_state=SEED,
)

print(f'Feature matrix X shape: {X.shape}')
print(f'Label vector y shape:   {y.shape}')
print(f'Class labels present:   {np.unique(y)}')
print(f'First example x_0:      {X[0]}')
print(f'Its label y_0:          {y[0]}')

## 3. Make a training split and an untouched test split

A 30% test split means about 300 of the 1,000 rows are held out; the remaining about 700 are for training. The fixed `random_state` makes the split reproducible. We do not alter either set's labels in this baseline cell. Later, when demonstrating poisoning, only a copy of the training labels should be changed. Keeping test labels correct lets us measure whether the model learned worse patterns from the corrupted training answers.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=SEED,
)

print(f'Training examples: {len(y_train)}')
print(f'Test examples:     {len(y_test)}')
print(f'Training class counts: {np.bincount(y_train)}')
print(f'Test class counts:     {np.bincount(y_test)}')

## 4. Plot the clean training data

Each dot is one training example. Its horizontal and vertical locations are the two features in `x_i`; its color comes from `y_i`. The plot shows the starting point before any label changes. Separate clusters make this classroom example easy to see, but real data may overlap and have many more dimensions.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))
fig.patch.set_facecolor('#141d2b')
ax.set_facecolor('#141d2b')

# X_train[:, 0] selects feature 1 for all training rows;
# X_train[:, 1] selects feature 2. The color is the unchanged label.
for class_id, color in enumerate(CLASS_COLORS):
    rows = y_train == class_id
    ax.scatter(
        X_train[rows, 0],
        X_train[rows, 1],
        color=color,
        edgecolors='#141d2b',
        alpha=0.8,
        label=f'Class {class_id}',
    )

ax.set_title('Clean training data before label flipping', color=HTB_GREEN)
ax.set_xlabel('Synthetic feature 1', color='white')
ax.set_ylabel('Synthetic feature 2', color='white')
ax.tick_params(colors='#a4b1cd')
ax.grid(alpha=0.2, color='#a4b1cd')
ax.legend()
plt.show()

## 5. Fit and evaluate the clean baseline

For example i, the model score is z_i = w^T x_i + b. Say it aloud: "z sub i equals w transpose times x sub i, plus b." The weights multiply the two feature values and the bias shifts the score. The sigmoid p_i = 1 / (1 + e^(-z_i)) converts this score to a probability of class 1. Scikit-learn learns its weights from clean training pairs only. We measure accuracy on the held-out test set, which was not used to fit the model.

In [ ]:
# Fit only on the original, clean training examples and labels.
baseline_model = LogisticRegression(random_state=SEED)
baseline_model.fit(X_train, y_train)

# Compare predictions with untouched, correct test labels.
y_pred_baseline = baseline_model.predict(X_test)
baseline_accuracy = accuracy_score(y_test, y_pred_baseline)
print(f'Baseline model test accuracy: {baseline_accuracy:.4f}')

# HTB reports 0.9933 for this fixed, well-separated toy dataset.
# Small library or solver-version differences can affect the last decimal.

## 6. Visualize the model decision boundary

The classifier predicts class 1 when p >= 0.5. Sigmoid is 0.5 when z = 0, so the boundary is w^T x + b = 0. With two features this is a straight line. To draw it, we predict classes at nearby feature coordinates and shade each region. This visualizes the fitted model; it does not train it again. We use grid spacing 0.05 instead of HTB 0.02 to reduce temporary points while keeping the plot clear.

In [ ]:
def plot_decision_boundary(model, X_plot, y_plot, title, mesh_step=0.05):
    """Draw predicted class regions and the two-feature examples."""
    # Add padding so edge examples are not clipped.
    x_min, x_max = X_plot[:, 0].min() - 1, X_plot[:, 0].max() + 1
    y_min, y_max = X_plot[:, 1].min() - 1, X_plot[:, 1].max() + 1

    # Build a lightweight grid of feature coordinates for prediction.
    xx, yy = np.meshgrid(
        np.arange(x_min, x_max, mesh_step),
        np.arange(y_min, y_max, mesh_step),
    )
    grid_points = np.column_stack((xx.ravel(), yy.ravel()))
    grid_predictions = model.predict(grid_points).reshape(xx.shape)

    fig, ax = plt.subplots(figsize=(9, 6))
    fig.patch.set_facecolor('#141d2b')
    ax.set_facecolor('#141d2b')
    ax.contourf(
        xx, yy, grid_predictions,
        levels=[-0.5, 0.5, 1.5], colors=CLASS_COLORS, alpha=0.3,
    )

    # Overlay the original examples, colored by their clean labels.
    for class_id, color in enumerate(CLASS_COLORS):
        rows = y_plot == class_id
        ax.scatter(
            X_plot[rows, 0], X_plot[rows, 1],
            color=color, edgecolors='#141d2b', alpha=0.8,
            label=f'Class {class_id}',
        )

    ax.set_title(title, color=HTB_GREEN)
    ax.set_xlabel('Synthetic feature 1', color='white')
    ax.set_ylabel('Synthetic feature 2', color='white')
    ax.tick_params(colors='#a4b1cd')
    ax.grid(alpha=0.2, color='#a4b1cd')
    ax.legend()
    plt.show()

# Visualize the model trained above; no additional fitting happens here.
plot_decision_boundary(
    baseline_model,
    X_train,
    y_train,
    title=f'Clean baseline decision regions (test accuracy {baseline_accuracy:.4f})',
)

## Check your understanding

1. In x_i = (x_i1, x_i2), which part is the label? (Neither; the label is y_i.)
2. In label flipping, what changes: the feature row, the label, or both? (The label changes; features stay fixed.)
3. Why keep test labels clean? (So evaluation measures the model against correct answers.)
4. In z = w^T x + b, what do the weights and bias do? (They form and shift the linear score.)
5. What does sigmoid do? (It maps a score to a probability between zero and one.)
6. Why establish a clean baseline? (So later model performance can be compared against a trusted reference.)
7. Does this plot and its high accuracy demonstrate a real sentiment system? (No. It uses artificial two-dimensional data.)

8. Why can accuracy remain unchanged even when a decision boundary moves? (The boundary can move without changing any held-out prediction.)
9. Why evaluate each poisoned model on the original clean test labels? (They are the trusted answer key.)
10. Does the 10% result prove 10% poisoning is harmless? (No; it is specific to this easy synthetic dataset.)

Scope note: this notebook covers Sections 3-6: the clean baseline, label flipping, and evaluation across multiple poisoning rates using a clean test set.

## 7. What a flipped label does to the training loss

The Section 4 binary cross-entropy loss for one example is

`-[y_i log(p_i) + (1 - y_i) log(1 - p_i)]`.

Read it aloud: **"negative, open bracket, y sub i times log of p sub i, plus one minus y sub i times log of one minus p sub i, close bracket."** `y_i` is the supplied 0/1 answer and `p_i` is the model's predicted probability of class 1. The label selects which probability the training process rewards.

Suppose a review truly has label `y_i = 0` and the model gives `p_i = 0.1`. With the correct label, the loss is `-log(1 - 0.1) = -log(0.9) ≈ 0.105`, a small penalty. If the label is flipped to `y_i' = 1`, the loss becomes `-log(0.1) ≈ 2.303`. Same features, same current prediction, but now the optimizer sees a much larger error and is pushed to change the model. This is the mechanism behind the attack: training optimizes against the labels it is given, even when some are false.

The poisoning percentage controls **how many** training labels change. It does not guarantee a particular accuracy drop; impact depends on which rows are selected, the dataset, and the model. We keep the test set clean so it remains a trustworthy measuring stick in the later comparison.

## 8. Implement `flip_labels`

This function receives labels and a poisoning fraction. It validates the fraction, converts it into a whole-number count, samples that many unique row indices, copies the labels, and flips only the selected entries. `1 - y` is correct here because the dataset uses binary labels 0 and 1: zero becomes one and one becomes zero.

The course code uses a seeded NumPy generator. This notebook accepts an explicit `seed` argument (defaulting to the lesson's `SEED`) so the random choice is reproducible without relying on hidden global state. Sampling **without replacement** means an index cannot be selected twice. Returning the indices lets us verify and visualize exactly which rows were changed.

In [ ]:
def flip_labels(y, poison_percentage, seed=SEED):
    """Return a copied label array with a reproducible subset flipped.

    The labels must be binary 0/1 values. The original array is never modified.
    The optional seed makes the random selection repeatable.
    """
    if not 0 <= poison_percentage <= 1:
        raise ValueError("poison_percentage must be between 0 and 1.")

    y = np.asarray(y)
    if not np.isin(y, [0, 1]).all():
        raise ValueError("This label-flipping function requires binary 0/1 labels.")

    n_samples = len(y)
    n_to_flip = int(n_samples * poison_percentage)
    if n_to_flip == 0:
        # For zero or a very small rate, no full example may be selected.
        return y.copy(), np.array([], dtype=int)

    rng_instance = np.random.default_rng(seed)
    flipped_indices = rng_instance.choice(
        n_samples,
        size=n_to_flip,
        replace=False,
    )

    # Work on a copy: y_train and the clean baseline stay intact.
    y_poisoned = y.copy()
    original_labels_at_flipped = y_poisoned[flipped_indices]
    y_poisoned[flipped_indices] = np.where(
        original_labels_at_flipped == 0,
        1,
        0,
    )
    return y_poisoned, flipped_indices

## 9. Visualize the altered labels

The feature matrix remains the same: each point stays at the same `(feature 1, feature 2)` location. Its fill color shows the label after the operation; a red `X` outline marks rows whose labels were flipped. The mask is a Boolean selector: it keeps rows not in `flipped_indices` for the ordinary dots. The second scatter call draws only the selected rows with a distinctive marker.

This visualization is a diagnostic aid. In real high-dimensional training data, the feature rows may not be plottable, so use the returned indices and controlled data checks instead.

In [ ]:
def plot_poisoned_data(X_plot, y_poisoned, flipped_indices, title):
    """Plot unchanged 2D features, coloring by new label and marking flips."""
    flipped_mask = np.zeros(len(y_poisoned), dtype=bool)
    flipped_mask[flipped_indices] = True
    unchanged_mask = ~flipped_mask

    fig, ax = plt.subplots(figsize=(9, 6))
    fig.patch.set_facecolor('#141d2b')
    ax.set_facecolor('#141d2b')

    # Ordinary points are colored by the label now supplied to training.
    for class_id, color in enumerate(CLASS_COLORS):
        rows = unchanged_mask & (y_poisoned == class_id)
        ax.scatter(
            X_plot[rows, 0], X_plot[rows, 1],
            color=color, edgecolors='#141d2b', alpha=0.55,
            label=f'Unchanged row, current label {class_id}',
        )

    # A red X identifies rows whose labels changed, whatever their new class.
    ax.scatter(
        X_plot[flipped_mask, 0], X_plot[flipped_mask, 1],
        c=y_poisoned[flipped_mask], cmap=plt.matplotlib.colors.ListedColormap(CLASS_COLORS),
        edgecolors='#ff5f5f', linewidths=1.5, marker='X', s=110,
        label='Flipped label',
    )
    ax.set_title(title, color=HTB_GREEN)
    ax.set_xlabel('Synthetic feature 1', color='white')
    ax.set_ylabel('Synthetic feature 2', color='white')
    ax.tick_params(colors='#a4b1cd')
    ax.grid(alpha=0.2, color='#a4b1cd')
    ax.legend()
    plt.show()

## 10. Poison a copy of the training labels

A 10% rate is a demonstration choice for this notebook, not a required rate from HTB. With 700 training rows, `int(700 * 0.10)` is 70. We flip only a **copy** of `y_train`; the original clean training labels and all test labels remain available for comparison later. This cell demonstrates the current attack step only—it does not train a poisoned model or claim an accuracy impact.

In [ ]:
POISON_PERCENTAGE = 0.10  # Demonstration setting, not an HTB-mandated value.
clean_train_labels = y_train.copy()
clean_test_labels = y_test.copy()
y_train_poisoned, flipped_indices = flip_labels(
    y_train,
    poison_percentage=POISON_PERCENTAGE,
    seed=SEED,
)

expected_flips = int(len(y_train) * POISON_PERCENTAGE)
print(f"Training labels selected: {len(flipped_indices)} / {len(y_train)}")
print(f"Requested rate: {POISON_PERCENTAGE:.0%}; actual whole-row rate: {len(flipped_indices) / len(y_train):.2%}")
print(f"Original training labels preserved: {np.array_equal(y_train, clean_train_labels)}")
print(f"Number of labels that differ: {np.count_nonzero(y_train_poisoned != y_train)}")

# Repeatability and scope checks: the same seed picks the same rows;
# features and clean test labels are not changed by this function.
repeat_labels, repeat_indices = flip_labels(y_train, POISON_PERCENTAGE, seed=SEED)
assert np.array_equal(y_train_poisoned, repeat_labels)
assert np.array_equal(flipped_indices, repeat_indices)
assert np.count_nonzero(y_train_poisoned != y_train) == expected_flips
assert np.array_equal(y_train, clean_train_labels)
assert np.array_equal(y_test, clean_test_labels)
assert len(y_test) == len(X_test)

plot_poisoned_data(
    X_train,
    y_train_poisoned,
    flipped_indices,
    title=f"Training labels after {POISON_PERCENTAGE:.0%} label flipping",
)

## 11. Measure accuracy at several poisoning rates

Let `q` mean the fraction of training labels changed. For each `q`, we train a fresh model on the original feature rows and labels modified at that rate. We then score it on the same clean held-out test set.

Accuracy can be written as

`accuracy(q) = (1 / N_test) * sum from i=1 to N_test of 1[model_q(x_i) = y_i]`.

Read this aloud: **“accuracy at q equals one over N test times the sum, from i equals one to N test, of the indicator that model q predicts x sub i equal to y sub i.”** `N_test` is the number of test examples. The indicator is a counting switch: it is 1 when the prediction matches the correct label and 0 when it does not. The sum counts correct predictions; dividing by the total turns the count into a fraction. For example, 297 correct predictions out of 300 gives `297 / 300 = 0.99`, or 99% accuracy.

The experiment uses the same rates HTB discusses: 0%, 10%, 20%, 30%, 40%, and 50%. At 0%, the clean training labels create the baseline. For every other rate, a new poisoned label copy is made. The test labels are never passed to the poisoning function.

In [ ]:
POISON_RATES = [0.0, 0.10, 0.20, 0.30, 0.40, 0.50]
models_by_rate = {}
labels_by_rate = {}
flips_by_rate = {}
accuracy_by_rate = {}

for poison_rate in POISON_RATES:
    if poison_rate == 0.0:
        # The zero-rate point is the clean baseline; keep a separate label copy.
        labels_for_fit = y_train.copy()
        flipped_for_rate = np.array([], dtype=int)
    else:
        labels_for_fit, flipped_for_rate = flip_labels(
            y_train,
            poison_percentage=poison_rate,
            seed=SEED,
        )

    # Fit a fresh classifier for this poisoning level.
    model = LogisticRegression(random_state=SEED)
    model.fit(X_train, labels_for_fit)

    # Evaluate only against the original, clean held-out labels.
    predictions = model.predict(X_test)
    score = accuracy_score(y_test, predictions)

    models_by_rate[poison_rate] = model
    labels_by_rate[poison_rate] = labels_for_fit
    flips_by_rate[poison_rate] = flipped_for_rate
    accuracy_by_rate[poison_rate] = score

    print(
        f"Poison rate {poison_rate:>4.0%}: "
        f"{len(flipped_for_rate):>3} labels changed; "
        f"clean-test accuracy = {score:.4f}"
    )

# Confirm the evaluation did not corrupt the original labels.
assert np.array_equal(y_train, clean_train_labels)
assert np.array_equal(y_test, clean_test_labels)

## 12. Plot accuracy and compare learned boundaries

The left plot shows measured accuracy versus the fraction of training labels changed. It summarizes only the predictions made on this fixed clean test set. The right plot overlays each model's boundary on one set of axes. A boundary can shift even when no test point crosses it, which is why the accuracy line and boundary plot answer related but different questions.

For the overlay, we make one grid of nearby feature coordinates, predict each grid point with every trained model, and draw a contour at class value 0.5. The original clean training points provide visual context. The plot is an approximation at finite grid spacing; it is not another training operation.

In [ ]:
fig, (ax_accuracy, ax_boundary) = plt.subplots(1, 2, figsize=(14, 6))
fig.patch.set_facecolor('#141d2b')
for ax in (ax_accuracy, ax_boundary):
    ax.set_facecolor('#141d2b')
    ax.tick_params(colors='#a4b1cd')
    ax.grid(alpha=0.2, color='#a4b1cd')

# Panel 1: clean-test accuracy as training-label corruption increases.
rate_percent = [rate * 100 for rate in POISON_RATES]
accuracy_values = [accuracy_by_rate[rate] for rate in POISON_RATES]
ax_accuracy.plot(rate_percent, accuracy_values, marker='o', color=HTB_GREEN)
ax_accuracy.set_title('Accuracy on the clean test set', color=HTB_GREEN)
ax_accuracy.set_xlabel('Training labels flipped (%)', color='white')
ax_accuracy.set_ylabel('Accuracy', color='white')
ax_accuracy.set_ylim(0, 1.05)
ax_accuracy.set_xticks(rate_percent)

# Panel 2: overlay boundaries from the same models.
x_min, x_max = X_train[:, 0].min() - 1, X_train[:, 0].max() + 1
y_min, y_max = X_train[:, 1].min() - 1, X_train[:, 1].max() + 1
mesh_step = 0.05
xx, yy = np.meshgrid(
    np.arange(x_min, x_max, mesh_step),
    np.arange(y_min, y_max, mesh_step),
)
mesh_points = np.column_stack((xx.ravel(), yy.ravel()))

for class_id, color in enumerate(CLASS_COLORS):
    rows = y_train == class_id
    ax_boundary.scatter(
        X_train[rows, 0], X_train[rows, 1],
        color=color, edgecolors='#141d2b', alpha=0.35, s=24,
    )

boundary_colors = ['#9fef00', '#00ffff', '#ffaf00', '#bd93f9', '#0086ff', '#ff5f5f']
for poison_rate, color in zip(POISON_RATES, boundary_colors):
    model = models_by_rate[poison_rate]
    grid_classes = model.predict(mesh_points).reshape(xx.shape)
    ax_boundary.contour(
        xx, yy, grid_classes,
        levels=[0.5], colors=[color], linewidths=2,
    )

ax_boundary.set_title('Boundary movement as poisoning rises', color=HTB_GREEN)
ax_boundary.set_xlabel('Synthetic feature 1', color='white')
ax_boundary.set_ylabel('Synthetic feature 2', color='white')
legend_handles = [
    plt.Line2D([0], [0], color=color, lw=2, label=f'{rate:.0%} poisoned')
    for rate, color in zip(POISON_RATES, boundary_colors)
]
ax_boundary.legend(handles=legend_handles, title='Training labels', facecolor='#141d2b', labelcolor='white')
fig.tight_layout()
plt.show()

### Read the result carefully

The HTB dataset is artificially easy to separate. At 10%, its clean-test accuracy remains `0.9933` even though the boundary shifts slightly. This means none of the held-out examples changed predicted class; it does **not** mean the learned model is identical. As the poisoning fraction grows, the boundaries continue to move, and at higher rates accuracy can fall. The exact curve depends on the selected rows, model, data distribution, and random seed.

For a real system, also inspect class-specific measures such as precision and recall, and consider whether the behavior of interest is captured by the test set. Here the goal is to demonstrate the evaluation method and the gap between a boundary change and a measured accuracy change—not to claim a universal threshold for label poisoning.